In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import os
import random
import numpy as np
import pandas as pd
from glob import glob
from tqdm.auto import tqdm

import torch
import torchaudio
import torchaudio.functional as F
import torchaudio.transforms as T
import librosa

import warnings
warnings.filterwarnings('ignore')
BASE_DIR = "/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup"

In [2]:
noise_paths = glob(f"{BASE_DIR}/ESC-50-master/audio/*.wav")
stem_files = glob(f"{BASE_DIR}/genres_stems/*/*/*.wav")

In [3]:
stem_records = []
for path in stem_files:
    parts = path.split('/')
    stem_records.append({
        "genre": parts[-3],
        "song_id": parts[-2],
        "stem_type": parts[-1].replace(".wav", ""),
        "file_path": path
    })

In [5]:
df_stems = pd.DataFrame(stem_records)
print(df_stems)

Empty DataFrame
Columns: []
Index: []


In [4]:
df_stems = pd.DataFrame(stem_records)
genres = sorted(df_stems["genre"].unique())
genre_to_label = {g: i for i, g in enumerate(genres)}
np.random.seed(42)

unique_songs = df_stems['song_id'].unique()
np.random.shuffle(unique_songs)

split_idx = int(len(unique_songs) * 0.8)

KeyError: 'genre'

In [ ]:
train_df = df_stems[df_stems['song_id'].isin(unique_songs[:split_idx])].copy()
val_df   = df_stems[df_stems['song_id'].isin(unique_songs[split_idx:])].copy()
sr = 22050
duration = 30
crop_sec = 10

In [ ]:
samples = sr * duration
crop_samples = sr * crop_sec

mel_transform = T.MelSpectrogram(sample_rate=sr, n_mels=128, n_fft=2048, hop_length=512)
amp_to_db = T.AmplitudeToDB(top_db=80)

time_mask = T.TimeMasking(time_mask_param=30)
freq_mask = T.FrequencyMasking(freq_mask_param=15)

In [ ]:
def load_audio(path):
    waveform, sample_rate = torchaudio.load(path)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0)
    else:
        waveform = waveform.squeeze(0)

    if sample_rate != sr:
        resampler = T.Resample(sample_rate, sr)
        waveform = resampler(waveform)

    return waveform.contiguous()

In [ ]:
def generate_robust_dataset(df, size=5000, is_train=True):
    genre_groups = {g: df[df["genre"]==g] for g in genres}
    X, y = [], []

    for _ in tqdm(range(size)):
        genre = random.choice(genres)
        label = genre_to_label[genre]
        genre_df = genre_groups[genre]

        mix = torch.zeros(samples)

        # 1. Stem Mixing
        for stem_type in ["bass", "drums", "vocals", "other"]:
            stem_subset = genre_df[genre_df["stem_type"]==stem_type]
            song = random.choice(stem_subset["song_id"].unique())
            stem_row = stem_subset[stem_subset["song_id"]==song].iloc[0]

            y_audio = load_audio(stem_row["file_path"])
            y_audio = y_audio[:samples]
            mix[:len(y_audio)] += y_audio

        max_val = torch.max(torch.abs(mix))
        if max_val > 0:
            mix = mix / (max_val + 1e-8)

        # 2. Audio-Level Augmentations (Only applied to training data)
        if is_train:
            # Time Shift (Updated to +/- 1 second rule)
            if random.random() < 0.5:
                shift_amount = int(sr * random.uniform(-1.0, 1.0))
                mix = torch.roll(mix, shifts=shift_amount, dims=0)

            # Pitch Shift
            if random.random() < 0.3:
                n_steps = random.randint(-2, 2)
                mix = F.pitch_shift(mix, sr, n_steps)

            # Time Stretch
            if random.random() < 0.3:
                speed_factor = random.uniform(0.85, 1.15)
                mix_np = mix.numpy()
                stretched = librosa.effects.time_stretch(mix_np, rate=speed_factor)
                mix = torch.from_numpy(stretched)

            # Add Noise (Split into Environmental vs Gaussian)
            if random.random() < 0.7:
                if random.random() < 0.5 and len(noise_paths) > 0:
                    # Environmental noise can be a bit louder (5% to 20%)
                    noise_strength = random.uniform(0.05, 0.20)
                    noise_path = random.choice(noise_paths)
                    noise = load_audio(noise_path)
                    if len(noise) < len(mix):
                        start = random.randint(0, len(mix)-len(noise))
                        mix[start:start+len(noise)] += noise_strength * noise
                else:
                    # Gaussian White Noise MUST be tiny (0.3% to 2%)
                    noise_strength = random.uniform(0.003, 0.02)
                    gaussian_noise = torch.randn_like(mix) * noise_strength
                    mix += gaussian_noise

        # 3. Smart Crop & Pad
        if len(mix) < crop_samples:
            mix = torch.nn.functional.pad(mix, (0, crop_samples - len(mix)))
            
        if len(mix) > crop_samples:
            frame_length = crop_samples
            hop = frame_length // 4
            mix_tensor = mix.unsqueeze(0).unsqueeze(0)
            energies = torch.nn.functional.avg_pool1d(mix_tensor ** 2, kernel_size=frame_length, stride=hop)
            best_start = torch.argmax(energies).item() * hop
            mix = mix[best_start:best_start + frame_length]

        # 4. Spectrogram Conversion
        mel = mel_transform(mix)
        mel = amp_to_db(mel)
        mel = (mel - mel.mean()) / (mel.std() + 1e-8)

        # 5. Spectrogram-Level Augmentations (SpecAugment)
        if is_train:
            if random.random() < 0.5:
                mel = time_mask(mel)
            if random.random() < 0.5:
                mel = freq_mask(mel)
        
        X.append(mel.unsqueeze(0))
        y.append(label)

    return torch.stack(X), torch.tensor(y)

In [ ]:
chunk_size = 5000
total_chunks = 4

for chunk in range(1, total_chunks + 1):
    X_chunk, y_chunk = generate_robust_dataset(train_df, size=chunk_size, is_train=True)
    
    torch.save(X_chunk, f"X_train_augmented_part{chunk}.pt")
    torch.save(y_chunk, f"y_train_augmented_part{chunk}.pt")

X_val, y_val = generate_robust_dataset(val_df, size=4000, is_train=False)

In [ ]:
torch.save(X_val, "X_val_clean.pt")
torch.save(y_val, "y_val_clean.pt")